# 04 — Multi-candidate verifier collection

Second round: 300 unique states × 4 candidates = 1,200 outcomes (about 24 L4 GPU-hours). Resumable and shardable; v1 data is preserved.

## 1. Environment setup

In [ ]:
import os, subprocess, sys
try:
    from google.colab import userdata
    for key in ("SUPABASE_URL", "SUPABASE_SERVICE_KEY", "HF_TOKEN", "WANDB_API_KEY"):
        value = userdata.get(key)
        if value:
            os.environ[key] = value
    repo_dir = "/content/cs159-sp26"
    gh_pat = userdata.get("GH_PAT")
    repo_url = f"https://{gh_pat}@github.com/ArjunS07/cs159-sp26.git"
    if not os.path.isdir(os.path.join(repo_dir, ".git")):
        subprocess.run(["git", "clone", "--branch", "main", repo_url, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only", "origin", "main"], check=True)
except ImportError:
    repo_dir = os.path.abspath("..") if os.path.basename(os.getcwd()) == "pnp-vla" else os.getcwd()

package_dir = os.path.join(repo_dir, "pnp-vla")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", package_dir + "[sim,analysis]"], check=True)
if package_dir not in sys.path:
    sys.path.insert(0, package_dir)
import pnp
print("Loaded pnp from:", pnp.__file__)

## 2. Load policy, store, and benchmark episode manifests

In [ ]:
import json
from tqdm.auto import tqdm
from pnp import libero_env, libero_pro, models
from pnp.experiments import _prepare_libero_pro_episodes
from pnp.store import SupabaseStore
from pnp.verifier import *

benchmark_dict = libero_env.init_libero_benchmark()
libero_pro.patch_torch_load()
policy, preprocess, postprocess = models.load_pi05()
device = models.default_device()
store = SupabaseStore()

suites = ("libero_spatial", "libero_object", "libero_goal", "libero_10")
tasks = [(suite, task) for suite in suites for task in range(benchmark_dict[suite]().n_tasks)]
standard = libero_env.build_final_episodes(benchmark_dict, tasks=tasks)
for ep in standard: ep["benchmark"] = "libero"
pro = _prepare_libero_pro_episodes()
for ep in pro: ep["benchmark"] = "libero_pro"
episode_lookup = {(e["benchmark"], e["suite"], e["task_idx"], e.get("ep_idx", e.get("episode_idx"))): e
                  for e in standard + pro}
print(len(standard), len(pro), len(episode_lookup))

## 3. Configure workers and build the fixed uncertainty manifest

In [ ]:
COLLECTION_EXPERIMENT = "verifier-clean-pairs-v2"
TARGETS = {"libero": 120, "libero_pro": 180}
CANDIDATE_COUNT = 4
PREFIX_LENGTH = 10
SHARD_COUNT = 1   # Use 3 for three parallel Colab sessions.
SHARD_INDEX = 0   # Set to 0, 1, or 2 in each session.
assert 0 <= SHARD_INDEX < SHARD_COUNT

def pages(table, columns, configure):
    rows=[]; start=0
    while True:
        q = configure(store.client.table(table).select(columns)).range(start, start+999)
        batch = q.execute().data or []; rows += batch
        if len(batch) < 1000: return rows
        start += 1000

experiments = ("libero-hybrid-schedules-k3-v1", "libero-pro-canonical-core-k3-v1")
rollouts=[]
for experiment in experiments:
    rollouts += pages("rollouts", "rollout_id,benchmark,suite,task_idx,episode_idx,success", lambda q, e=experiment:
        q.eq("experiment", e).eq("method", "pnp_uncertainty_only").eq("status", "completed"))
ids = {r["rollout_id"] for r in rollouts}
euler = []
id_list = sorted(ids)
for start in range(0, len(id_list), 100):
    batch_ids = id_list[start:start+100]
    euler += pages("pnp_euler_steps", "rollout_id,chunk_idx,u_mean",
                   lambda q, batch_ids=batch_ids: q.in_("rollout_id", batch_ids))
manifest = build_stratified_manifest(rollouts, euler, TARGETS)
TOTAL_GROUPS = len(manifest)
if TOTAL_GROUPS < sum(TARGETS.values()):
    print(f"eligible unique-state capacity: {TOTAL_GROUPS}/{sum(TARGETS.values())}; "
          "collecting all available groups")
manifest = manifest[SHARD_INDEX::SHARD_COUNT]
print({k: sum(r["benchmark"] == k for r in manifest) for k in ("libero", "libero_pro")})
print({k: sum(r["uncertainty_stratum"] == k for r in manifest) for k in ("low", "mid", "high")})

## 4. Dry-run identity and schema checks

In [ ]:
missing = [row for row in manifest if (row["benchmark"], row["suite"], row["task_idx"], row["episode_idx"]) not in episode_lookup]
assert not missing, missing[:3]
store.client.table("verifier_candidate_groups").select("candidate_group_id").limit(1).execute()
print("manifest identities and verifier tables are ready")

## 5. Collect resumable four-candidate groups

In [ ]:
existing = {r["candidate_group_id"] for r in
            (store.client.table("verifier_candidate_groups").select("candidate_group_id")
             .eq("experiment", COLLECTION_EXPERIMENT).execute().data or [])}
store.start_run("verifier_pair_collection", "libero+libero_pro", COLLECTION_EXPERIMENT,
                config={"groups": TOTAL_GROUPS,
                        "outcomes": TOTAL_GROUPS * CANDIDATE_COUNT,
                        "candidate_count": CANDIDATE_COUNT, "prefix_length": PREFIX_LENGTH,
                        "shard_count": SHARD_COUNT, "shard_index": SHARD_INDEX})
completed = 0
for item in tqdm(manifest, desc="candidate groups"):
    ep = episode_lookup[(item["benchmark"], item["suite"], item["task_idx"], item["episode_idx"])]
    expected_id = candidate_group_id(item["benchmark"], item["suite"], item["task_idx"],
                                     item["episode_idx"], item["chunk_idx"],
                                     namespace=COLLECTION_EXPERIMENT)
    fallback_id = candidate_group_id(item["benchmark"], item["suite"], item["task_idx"],
                                     item["episode_idx"], 0,
                                     namespace=COLLECTION_EXPERIMENT)
    if expected_id in existing or fallback_id in existing:
        continue
    env = libero_env.make_env(ep["bddl_path"])
    try:
        try:
            pair = collect_candidate_pair(
                env, ep, policy, preprocess, postprocess, device,
                chunk_idx=item["chunk_idx"], uncertainty_stratum=item["uncertainty_stratum"],
                prefix_length=PREFIX_LENGTH, validate_snapshot=True,
                candidate_count=CANDIDATE_COUNT, experiment=COLLECTION_EXPERIMENT)
        except Exception as error:
            print("snapshot fallback:", error)
            pair = collect_initial_pair_fallback(
                env, ep, policy, preprocess, postprocess, device,
                uncertainty_stratum=item["uncertainty_stratum"], prefix_length=PREFIX_LENGTH,
                source_chunk_idx=item["chunk_idx"], candidate_count=CANDIDATE_COUNT,
                experiment=COLLECTION_EXPERIMENT,
                fallback_reason=f"{type(error).__name__}: {error}")
        if pair is None:
            pair = collect_initial_pair_fallback(
                env, ep, policy, preprocess, postprocess, device,
                uncertainty_stratum=item["uncertainty_stratum"], prefix_length=PREFIX_LENGTH,
                source_chunk_idx=item["chunk_idx"], candidate_count=CANDIDATE_COUNT,
                experiment=COLLECTION_EXPERIMENT,
                fallback_reason="vanilla replay terminated before requested chunk")
        group, candidates = pair
        store.register_candidate_group(group, candidates)
        existing.add(group["candidate_group_id"]); completed += len(candidates)
    finally:
        env.close()
store.finish_run(n_rollouts=completed)
print("new outcomes:", completed, "total groups:", len(existing))

## 6. Integrity and outcome-balance report

In [ ]:
groups = store.client.table("verifier_candidate_groups").select("*").eq(
    "experiment", COLLECTION_EXPERIMENT).execute().data or []
candidates = store.client.table("verifier_candidates").select(
    "candidate_id,candidate_group_id,candidate_kind,success").execute().data or []
group_ids = {g["candidate_group_id"] for g in groups}
candidates = [c for c in candidates if c["candidate_group_id"] in group_ids]
print({"groups": len(groups), "outcomes": len(candidates),
       "successes": sum(c["success"] for c in candidates),
       "failures": sum(not c["success"] for c in candidates),
       "discordant_groups": sum(len({c["success"] for c in candidates if c["candidate_group_id"] == gid}) == 2
                                for gid in group_ids),
       "snapshot_groups": sum(g["pairing_mode"] == "snapshot" for g in groups),
       "fallback_groups": sum(g["pairing_mode"] != "snapshot" for g in groups)})